In [1]:
# ✅ Install library tambahan (optional jika sudah ada)
!pip install tensorflow tensorflowjs

In [2]:
import tensorflow as tf
import numpy as np
import os

# Contoh Model: MNIST Classifier
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=[28, 28]),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(loss="sparse_categorical_crossentropy",
              optimizer="adam",
              metrics=["accuracy"])

model.fit(X_train, y_train, epochs=3, validation_split=0.1)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.8719 - loss: 0.4542 - val_accuracy: 0.9652 - val_loss: 0.1264
Epoch 2/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 14s 8ms/step - accuracy: 0.9599 - loss: 0.1347 - val_accuracy: 0.9730 - val_loss: 0.0995
Epoch 3/3
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9735 - loss: 0.0902 - val_accuracy: 0.9748 - val_loss: 0.0926


In [3]:
# ✅ Export model ke SavedModel format
model_name = "my_mnist_model"
model_version = "0001"
export_dir = os.path.join(model_name, model_version)

tf.saved_model.save(model, export_dir)
print("✅ Model exported to:", export_dir)

✅ Model exported to: my_mnist_model/0001


In [4]:
# ✅ Convert ke TensorFlow Lite (TFLite)
converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)
tflite_model = converter.convert()

with open("mnist_model.tflite", "wb") as f:
    f.write(tflite_model)

print("✅ Model saved as TFLite: mnist_model.tflite")

✅ Model saved as TFLite: mnist_model.tflite


In [5]:
# ✅ Convert ke TensorFlow.js
!mkdir -p tfjs_model
!tensorflowjs_converter --input_format=tf_saved_model {export_dir} tfjs_model

print("✅ Model converted to TensorFlow.js format in ./tfjs_model")

2025-06-20 16:53:10.437838: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750438390.465000    2241 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750438390.472985    2241 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
🌲 Try https://ydf.readthedocs.io, the successor of TensorFlow Decision Forests with more features and faster training!
2025-06-20 16:53:18.109443: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
I0000 00:00:1750438398.292502    2241 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 0
I0000 00:00

In [6]:
## ✅ Hasil:
- SavedModel → `my_mnist_model/0001`
- TFLite → `mnist_model.tflite`
- TensorFlow.js → `tfjs_model/`

SyntaxError: invalid character '→' (U+2192) (ipython-input-6-1804311490.py, line 2)

# 📚 Chapter 19 - Training and Deploying TensorFlow Models at Scale

---

## 🔸 1. Tujuan Deployment
✅ Setelah model machine learning selesai dilatih, **deployment** dibutuhkan agar model dapat digunakan oleh aplikasi atau pengguna lain secara real-time.

---

## 🔸 2. Format Model TensorFlow
| Format        | Fungsi                           |
| ------------- | -------------------------------- |
| **SavedModel**| Format utama → untuk deployment  |
| **HDF5 (.h5)**| Format lama → model Keras        |
| **TFLite**    | Optimasi → untuk perangkat mobile/IoT |
| **TF.js**     | Untuk penggunaan di browser via JavaScript |

---

## 🔸 3. TensorFlow Serving
✅ Tool untuk **serving model TensorFlow dalam skala produksi**:
- Dibangun dengan **gRPC & REST API**
- Bisa digunakan dengan **Docker**

Contoh perintah:
```bash
docker run -p 8501:8501 --name tf_serving_mnist \
-v "/path/to/exported_model:/models/mnist" \
-e MODEL_NAME=mnist -t tensorflow/serving
```

----

## 🔸 4. Google Cloud AI Platform
✅ Layanan cloud untuk deployment model → mendukung SavedModel secara langsung.

Proses:

- Upload model ke Google Cloud Storage (GCS)

- Deploy model ke AI Platform

- Query model via REST API
---
## 🔸 5. TensorFlow Lite (TFLite)
✅ Untuk mobile & embedded devices

✅ File ukuran kecil

✅ Optimasi → quantization, pruning

✅ Dijalankan via TensorFlow Lite Interpreter
---
## 🔸 6. TensorFlow.js
✅ Untuk deployment model langsung ke browser

- File → bisa dihosting di server/web → dipanggil dengan JavaScript

- Mendukung training lanjutan di sisi client
---
## ✅ Kesimpulan
✅ SavedModel → format standar deployment

✅ TF Serving → produksi skala besar → docker/gRPC/REST

✅ TFLite → mobile/IoT

✅ TensorFlow.js → web applications

✅ Google Cloud AI Platform → deployment production di cloud